# Disney Exploration

In [14]:
import os
import sys
from pathlib import Path
import importlib

import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from stampli.paths import get_enriched_path
from stampli.util.runtime import bootstrap_spark_env

bootstrap_spark_env()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower

import stampli.util.display
importlib.reload(stampli.util.display)
from stampli.util.display import display_scrollable_dataframe
from stampli.paths import get_enriched_path


In [2]:
# Add src to sys.path so we can import stampli package
notebook_dir = Path("__file__").parent.resolve() if "__file__" in locals() else Path(".").resolve()
src_path = str(notebook_dir.parents[0]) # src/stampli -> src
if src_path not in sys.path:
    sys.path.append(src_path)
print(f"Added {src_path} to sys.path")

ENRICHED_PATH = str(get_enriched_path())


In [3]:

bootstrap_spark_env()

spark = (SparkSession.builder
    .appName("DisneyExploration")
    .config("spark.executor.memory", "16g")
    .config("spark.driver.memory", "16g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created (16GB)")

In [4]:
# column_order = [
#     'Branch', 'Rating', 'sentiment_score', 'sentiment_label',
#     'is_complaint', 'crowd_level', 'staff_sentiment', 'price_sensitivity',
#     'family_sentiment', 'Reviewer_Location', 'topics', 'Review_Text', 'Year_Month', 'review_uid', 
# ]

# read enriched data in the order of columns

spd_disney = (spark.read
    .parquet(ENRICHED_PATH)
    # .select(column_order)
)
disney_count = spd_disney.count()
print(f"Total Enriched Rows: {disney_count}")

In [5]:
# 5. Create Pandas DataFrame (pd_disney)

print("Converting to Pandas (pd_disney)...")
pd_disney = spd_disney.toPandas()

print(f"Pandas DF Shape: {pd_disney.shape}")
display_scrollable_dataframe(pd_disney)

In [6]:
pd_disney.columns

## Verification Blocks

In [7]:
# Check for 'Crowded' / 'Packed' reviews in specific locations
print("Crowd Level Distribution:")
print(pd_disney.groupby('Branch')['crowd_level'].value_counts())

In [9]:
# Keyword Check vs Extraction
KEYWORDS = ["crowd", "packed", "busy", "line", "queue", "wait", "full", "people"]

# Filter for reviews with keywords
mask = pd_disney['Review_Text'].str.contains('|'.join(KEYWORDS), case=False, na=False)
subset = pd_disney[mask]

print(f"Reviews with crowd keywords: {len(subset)}")

print("Of which have extracted crowd_level:")
print(subset['crowd_level'].value_counts(dropna=False))

# Show mis-matches (Keyword present, but crowd_level is NaN)
missed = subset[subset['crowd_level'].isna()]
if not missed.empty:
    print(f"\nPotential Misses ({len(missed)}):")
    display_scrollable_dataframe(missed[['Branch', 'Review_Text']])

### Theme x Sentiment Heatmap (Insight 1)
Using robust metrics (Median, Share Negative) gated by N>=30 reviews.

In [ ]:
from stampli.disney_viz import analyze_insight_1
# Call the analysis function (default save_dir=None displays plots inline)
analyze_insight_1(pd_disney)